# PY-07 | GeoPandas και σύνδεση στατιστικών με γεωμετρίες

Σε αυτό το μάθημα προσθέτουμε τη γεωγραφική διάσταση στο πραγματικό dataset της ΕΛΣΤΑΤ για την ανεργία του 2021.

Θα συνδέσουμε:

- τα όρια των **333 Δήμων** από το [ELSTAT — Δήμοι 2021 (DHMOI_2021.zip)](https://www.statistics.gr/documents/20181/1194366/DHMOI_2021.zip/55e3ba79-3475-70b8-5f8a-b912eeee389c)
και
- το processed CSV του PY-05 με **333 δημοτικές εγγραφές**.

> **Κεντρική ιδέα:** ένα table γίνεται γεωγραφικό dataset όταν κάθε εγγραφή συνδέεται με τη σωστή γεωμετρία μέσω ενός σταθερού κοινού αναγνωριστικού.

## 1. Στόχοι του μαθήματος

Στο τέλος του PY-07 θα μπορούμε να:

- εξηγούμε τη διαφορά `DataFrame` και `GeoDataFrame`,
- διαβάζουμε shapefile με `gpd.read_file()`,
- επιθεωρούμε τη στήλη `geometry`, το CRS και τους τύπους γεωμετρίας,
- δημιουργούμε έναν πρώτο χάρτη με `.plot()`,
- κατασκευάζουμε κοινό τετραψήφιο κωδικό Δήμου,
- ελέγχουμε τα κλειδιά πριν και μετά από ένα attribute join,
- συνδέουμε στατιστικά με γεωμετρίες μέσω `merge()`,
- δημιουργούμε θεματικό χάρτη ανεργίας,
- εξάγουμε το αποτέλεσμα σε GeoPackage.

## 2. Imports και paths

Απαιτούνται τα `pandas`, `geopandas` και `matplotlib`. Αν λείπουν:

```bash
python -m pip install pandas geopandas matplotlib
```

Αναμενόμενη δομή project:

```text
Data/
├── raw/
│   └── DHMOI_2021.zip
└── processed/
    └── elstat_unemployment_2021_municipalities.csv
```

In [ ]:
from pathlib import Path
from zipfile import ZipFile

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

BOUNDARY_ZIP = Path("Data/raw/DHMOI_2021.zip")
BOUNDARY_DIR = Path("Data/raw/DHMOI_2021")
CSV_PATH = Path(
    "Data/processed/elstat_unemployment_2021_municipalities.csv"
)
OUTPUT_PATH = Path(
    "Data/processed/elstat_unemployment_2021_municipalities.gpkg"
)

## 3. Προετοιμασία του boundary archive

Ένα shapefile δεν είναι ένα μόνο αρχείο: τα `.shp`, `.dbf`, `.shx`, `.prj` και άλλα συνοδευτικά αρχεία λειτουργούν μαζί.

Γνωρίζουμε ότι το boundary layer που χρειαζόμαστε ονομάζεται **`Municipalities2021.shp`**. Εξάγουμε λοιπόν το ZIP μόνο όταν χρειάζεται και αναζητούμε αυτό το συγκεκριμένο αρχείο αναδρομικά μέσα στον φάκελο, ώστε ο κώδικας να λειτουργεί ακόμη κι αν το ZIP δημιουργήσει υποφάκελο.

In [ ]:
if not BOUNDARY_ZIP.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε το boundary ZIP: {BOUNDARY_ZIP}"
    )

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε το processed CSV: {CSV_PATH}"
    )

shapefile_matches = list(
    BOUNDARY_DIR.rglob("Municipalities2021.shp")
)

if not shapefile_matches:
    BOUNDARY_DIR.mkdir(parents=True, exist_ok=True)

    with ZipFile(BOUNDARY_ZIP) as archive:
        archive.extractall(BOUNDARY_DIR)

    shapefile_matches = list(
        BOUNDARY_DIR.rglob("Municipalities2021.shp")
    )

if not shapefile_matches:
    raise FileNotFoundError(
        f"Δεν βρέθηκε το Municipalities2021.shp μέσα στο {BOUNDARY_DIR}"
    )

SHAPEFILE_PATH = shapefile_matches[0]

print("Shapefile:", SHAPEFILE_PATH)

## 4. Τι είναι ένα GeoDataFrame;

Ένα `pandas.DataFrame` αποθηκεύει γραμμές και στήλες. Ένα `GeoDataFrame` προσθέτει:

- ενεργή στήλη `geometry`,
- τύπους γεωμετρίας όπως `Point`, `LineString`, `Polygon` και `MultiPolygon`,
- σύστημα αναφοράς συντεταγμένων (`CRS`),
- γεωχωρικές μεθόδους και plotting.

Κάθε γραμμή του boundary layer αντιστοιχεί σε έναν Δήμο και η γεωμετρία της περιγράφει την έκτασή του.

## 5. Ανάγνωση του shapefile με GeoPandas

In [ ]:
municipalities = gpd.read_file(SHAPEFILE_PATH)

type(municipalities)

In [ ]:
municipalities.head()

## 6. Επιθεώρηση της πραγματικής δομής

Στο συγκεκριμένο αρχείο περιμένουμε **333 γραμμές και 8 στήλες**:

`Municip`, `OID_1`, `CODE`, `NAME_GR`, `NAME_ENG`, `Shape_Leng`, `Shape_Area`, `geometry`.

Δεν αρκεί να εμπιστευόμαστε την περιγραφή ενός αρχείου· ελέγχουμε πάντα το πραγματικό schema.

In [ ]:
print("Shape:", municipalities.shape)
print("Columns:", municipalities.columns.tolist())

municipalities.info()

In [ ]:
expected_boundary_columns = {
    "Municip", "OID_1", "CODE", "NAME_GR", "NAME_ENG",
    "Shape_Leng", "Shape_Area", "geometry",
}

assert len(municipalities) == 333
assert expected_boundary_columns.issubset(municipalities.columns)
assert municipalities["CODE"].nunique() == 333

## 7. Η στήλη `geometry`

Η `geometry` είναι ενεργή γεωμετρική στήλη. Οι τιμές της είναι αντικείμενα Shapely και όχι απλό κείμενο.

In [ ]:
print("Active geometry column:", municipalities.geometry.name)
print("First geometry type:", type(municipalities.geometry.iloc[0]))

municipalities.geometry.head(3)

## 8. Polygon και MultiPolygon

- `Polygon`: μία ενιαία πολυγωνική γεωμετρία.
- `MultiPolygon`: πολλά χωριστά πολυγωνικά τμήματα στην ίδια εγγραφή.

Οι νησιωτικοί ή γεωγραφικά κατακερματισμένοι Δήμοι συχνά χρειάζονται `MultiPolygon`.

In [ ]:
geometry_types = municipalities.geometry.geom_type

geometry_counts = geometry_types.value_counts()
geometry_counts

In [ ]:
assert geometry_counts.to_dict() == {
    "Polygon": 206,
    "MultiPolygon": 127,
}

print("Valid geometries:", municipalities.geometry.is_valid.sum())
print("Empty geometries:", municipalities.geometry.is_empty.sum())
print("Missing geometries:", municipalities.geometry.isna().sum())

assert municipalities.geometry.is_valid.all()
assert not municipalities.geometry.is_empty.any()
assert not municipalities.geometry.isna().any()

## 9. CRS: τι σημαίνει `EPSG:2100`;

Το CRS περιγράφει πώς οι αριθμητικές συντεταγμένες συνδέονται με θέσεις στη Γη. Το layer χρησιμοποιεί **EPSG:2100 — Greek Grid**, ένα προβολικό CRS κατάλληλο για την Ελλάδα, με μονάδα το μέτρο.

Δεν αλλάζουμε CRS χωρίς λόγο. Πριν από υπολογισμούς απόστασης ή έκτασης ελέγχουμε πάντα το CRS και τις μονάδες του.

In [ ]:
crs = municipalities.crs

print(crs)
print("Projected:", crs.is_projected)
print("EPSG:", crs.to_epsg())

first_axis = crs.axis_info[0]
print("Units:", first_axis.unit_name)

assert crs.to_epsg() == 2100

## 10. Πρώτος χάρτης με `.plot()`

Ο πρώτος χάρτης είναι έλεγχος δεδομένων: μας βοηθά να εντοπίσουμε απουσίες ή εμφανώς προβληματικές γεωμετρίες.

In [ ]:
ax = municipalities.plot(
    figsize=(9, 9),
    facecolor="#d9eaf7",
    edgecolor="#4d4d4d",
    linewidth=0.35,
)

ax.set_title("Municipalities of Greece, 2021")
ax.set_axis_off()
plt.show()

## 11. Φόρτωση και επιθεώρηση του CSV

Το `geo_code` διαβάζεται ως string ώστε να αντιμετωπίζεται ως αναγνωριστικό και όχι ως αριθμητικό μέγεθος.

In [ ]:
unemployment = pd.read_csv(
    CSV_PATH,
    dtype={"geo_code": "string"},
)

print("Shape:", unemployment.shape)
print("Columns:", unemployment.columns.tolist())
unemployment.info()

In [ ]:
unemployment.head()

In [ ]:
expected_csv_columns = {
    "geo_code",
    "municipality",
    "population_total",
    "economically_active",
    "employed_total",
    "unemployed",
    "unemployment_rate",
    "economically_inactive",
    "dominant_unemployed_education",
}

assert len(unemployment) == 333
assert expected_csv_columns.issubset(unemployment.columns)
assert unemployment["geo_code"].nunique() == 333
assert unemployment["geo_code"].str.len().eq(7).all()

## 12. Σύγκριση γεωγραφικών αναγνωριστικών

Το boundary layer έχει τετραψήφιο `CODE`, ενώ το CSV έχει επταψήφιο `geo_code`. Στις εγγραφές δημοτικού επιπέδου τα τελευταία τέσσερα ψηφία του `geo_code` αντιστοιχούν στον κωδικό Δήμου.

Παράδειγμα:

```text
geo_code: 1110101
last 4:      0101
CODE:        0101
```

Η σύνδεση θα γίνει με κωδικούς, όχι με ονόματα. Τα ονόματα είναι χρήσιμα για ανθρώπινο έλεγχο, αλλά μπορεί να αλλάζουν σε ορθογραφία, σημεία στίξης ή μορφοποίηση.

In [ ]:
municipalities[["CODE", "NAME_GR"]].head()

In [ ]:
unemployment[["geo_code", "municipality"]].head()

## 13. Δημιουργία κοινού κωδικού Δήμου

Δημιουργούμε νέα στήλη `municipality_code` και στα δύο datasets. Το `.copy()` αποφεύγει ανεπιθύμητες αλλαγές στα αρχικά αντικείμενα.

In [ ]:
municipalities_join = municipalities.copy()
unemployment_join = unemployment.copy()

boundary_code = municipalities_join["CODE"].astype("string")
boundary_code = boundary_code.str.strip()
boundary_code = boundary_code.str.zfill(4)

municipalities_join["municipality_code"] = boundary_code

statistics_code = unemployment_join["geo_code"].str.strip()
statistics_code = statistics_code.str[-4:]

unemployment_join["municipality_code"] = statistics_code

municipalities_join[
    ["CODE", "municipality_code"]
].head()

## 14. Validation πριν από το join

Πριν χρησιμοποιήσουμε `merge()` ελέγχουμε:

- missing values στο key,
- duplicates,
- πλήθος μοναδικών κωδικών,
- διαφορές μεταξύ των δύο code sets.

Έτσι ένα λάθος join αποτυγχάνει νωρίς και καθαρά.

In [ ]:
boundary_codes = set(municipalities_join["municipality_code"])
statistics_codes = set(unemployment_join["municipality_code"])

only_in_boundaries = boundary_codes - statistics_codes
only_in_statistics = statistics_codes - boundary_codes

print("Boundary codes:", len(boundary_codes))
print("Statistics codes:", len(statistics_codes))
print("Only in boundaries:", sorted(only_in_boundaries))
print("Only in statistics:", sorted(only_in_statistics))

assert not municipalities_join["municipality_code"].isna().any()
assert not unemployment_join["municipality_code"].isna().any()
assert not municipalities_join["municipality_code"].duplicated().any()
assert not unemployment_join["municipality_code"].duplicated().any()
assert boundary_codes == statistics_codes
assert len(boundary_codes) == 333

## 15. Attribute join με `merge()`

Χρησιμοποιούμε `left` join ώστε να διατηρήσουμε όλες τις γεωμετρίες. Το `validate="one_to_one"` δηλώνει τη σχέση που περιμένουμε και προκαλεί σαφές error αν κάποιο key επαναλαμβάνεται.

Το `indicator=True` δημιουργεί προσωρινά τη στήλη `_merge`, ώστε να δούμε αν κάθε γραμμή βρήκε ταίρι.

In [ ]:
municipal_unemployment = municipalities_join.merge(
    unemployment_join,
    on="municipality_code",
    how="left",
    validate="one_to_one",
    indicator=True,
)

type(municipal_unemployment)

In [ ]:
municipal_unemployment["_merge"].value_counts()

## 16. Validation μετά το join

Ένα join που εκτελείται χωρίς error δεν είναι απαραίτητα σωστό. Ελέγχουμε το πλήθος γραμμών, τα unmatched records, τις missing τιμές στον δείκτη και τη διατήρηση της γεωγραφικής πληροφορίας.

In [ ]:
merge_status = municipal_unemployment["_merge"]

unmatched = municipal_unemployment.loc[
    merge_status != "both",
    ["municipality_code", "NAME_GR", "municipality", "_merge"],
]

joined_rows = len(municipal_unemployment)
matched_rows = merge_status.eq("both").sum()
unmatched_rows = len(unmatched)
missing_rates = municipal_unemployment["unemployment_rate"].isna().sum()

print("Joined rows:", joined_rows)
print("Matched rows:", matched_rows)
print("Unmatched rows:", unmatched_rows)
print("Missing unemployment rates:", missing_rates)

assert isinstance(municipal_unemployment, gpd.GeoDataFrame)
assert joined_rows == 333
assert merge_status.eq("both").all()
assert municipal_unemployment["unemployment_rate"].notna().all()
assert municipal_unemployment.crs.to_epsg() == 2100

municipal_unemployment = municipal_unemployment.drop(
    columns="_merge"
)

In [ ]:
municipal_unemployment[
    [
        "municipality_code",
        "NAME_GR",
        "municipality",
        "unemployment_rate",
        "geometry",
    ]
].head()

## 17. Έλεγχος ονομάτων

Οι κωδικοί είναι το πραγματικό join key. Παρ' όλα αυτά, η σύγκριση των δύο στηλών ονόματος είναι ένας χρήσιμος δεύτερος έλεγχος για πιθανές ασυμφωνίες περιεχομένου.

In [ ]:
boundary_names = municipal_unemployment["NAME_GR"].str.strip()
statistics_names = municipal_unemployment["municipality"].str.strip()

name_mismatches = municipal_unemployment.loc[
    boundary_names != statistics_names,
    ["municipality_code", "NAME_GR", "municipality"],
]

print("Name mismatches:", len(name_mismatches))
name_mismatches.head()

## 18. Θεματικός χάρτης ποσοστού ανεργίας

Τώρα η κάθε γεωμετρία έχει το `unemployment_rate` του σωστού Δήμου. Με `column=` ζητάμε χρώμα βάσει τιμής και με `legend=True` εμφανίζουμε υπόμνημα.

Ο χάρτης δείχνει χωρικά patterns, αλλά δεν εξηγεί από μόνος του τις αιτίες τους.

In [ ]:
ax = municipal_unemployment.plot(
    column="unemployment_rate",
    cmap="OrRd",
    legend=True,
    figsize=(10, 10),
    edgecolor="white",
    linewidth=0.25,
    legend_kwds={"label": "Unemployment rate (%)"},
)

ax.set_title("Municipal unemployment rate, Greece, 2021")
ax.set_axis_off()
plt.show()

## 19. Επιθεώρηση του τελικού GeoDataFrame

Το τελικό αντικείμενο συνδυάζει:

- boundary attributes,
- στατιστικές μεταβλητές,
- κοινό key,
- ενεργή geometry,
- CRS.

In [ ]:
print("Type:", type(municipal_unemployment))
print("Shape:", municipal_unemployment.shape)
print("CRS:", municipal_unemployment.crs)
print("Geometry:", municipal_unemployment.geometry.name)

municipal_unemployment[
    ["municipality_code", "NAME_GR", "unemployment_rate", "geometry"]
].head()

## 20. Export σε GeoPackage

Το GeoPackage (`.gpkg`) αποθηκεύει γεωμετρίες, attributes και CRS σε ένα αρχείο. Είναι συνήθως πιο πρακτικό από ένα νέο shapefile και δεν έχει το ίδιο αυστηρό όριο μήκους στα column names.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

municipal_unemployment.to_file(
    OUTPUT_PATH,
    layer="municipal_unemployment_2021",
    driver="GPKG",
)

print(f"Saved to: {OUTPUT_PATH}")

In [ ]:
export_check = gpd.read_file(
    OUTPUT_PATH,
    layer="municipal_unemployment_2021",
)

assert len(export_check) == 333
assert export_check.crs.to_epsg() == 2100
assert export_check["unemployment_rate"].notna().all()

print("Export verified:", export_check.shape)

## Καλή πρακτική για παρόχους γεωχωρικών και στατιστικών δεδομένων

Η σύνδεση γίνεται πολύ πιο ασφαλής όταν ο πάροχος διαθέτει:

- κοινό, σταθερό geographic identifier σε boundaries και statistics,
- τον κωδικό ως string με τεκμηριωμένα leading zeros,
- ρητή έκδοση και έτος της διοικητικής γεωγραφίας,
- επίσημο concordance όταν οι κωδικοί ή τα όρια αλλάζουν,
- machine-readable schema και metadata,
- σαφές CRS και τεκμηρίωση της γεωμετρίας.

Τα ονόματα παραμένουν χρήσιμα για εμφάνιση και ποιοτικό έλεγχο, αλλά δεν πρέπει να είναι το μοναδικό join key.

## 21. Ασκήσεις

### Άσκηση 1

Βρες πόσοι Δήμοι έχουν `Polygon` και πόσοι `MultiPolygon`.

### Άσκηση 2

Δημιούργησε GeoDataFrame με τους 10 Δήμους που έχουν το υψηλότερο `unemployment_rate` και εμφάνισε `NAME_GR`, `unemployment_rate` και `geometry`.

### Άσκηση 3

Δημιούργησε χάρτη της μεταβλητής `population_total` με διαφορετικό colormap.

### Άσκηση 4

Επανάλαβε το join χωρίς `indicator=True`. Ποιον σημαντικό διαγνωστικό έλεγχο χάνεις;

### Άσκηση 5

Υπολόγισε την έκταση κάθε Δήμου σε τετραγωνικά χιλιόμετρα από τη `geometry` και βρες τους 5 μεγαλύτερους. Γιατί το EPSG:2100 είναι κατάλληλο για αυτόν τον υπολογισμό;

In [ ]:
# Άσκηση 1

In [ ]:
# Άσκηση 2

In [ ]:
# Άσκηση 3

In [ ]:
# Άσκηση 4

In [ ]:
# Άσκηση 5

## 22. Σύνοψη

Στο PY-07 δημιουργήσαμε ένα πλήρες, ελεγχόμενο attribute-join workflow:

```text
read boundaries + inspect geometry/CRS
                  ↓
read statistics + inspect schema
                  ↓
standardize stable municipality key
                  ↓
validate keys and code sets
                  ↓
one-to-one attribute join
                  ↓
validate 333/333 matches
                  ↓
map + export + verify
```

Το σημαντικότερο αποτέλεσμα δεν είναι μόνο ο χάρτης. Είναι ότι μπορούμε να αποδείξουμε πως και οι **333 από τις 333** γεωμετρίες συνδέθηκαν με τη σωστή στατιστική εγγραφή.

## 23. Λύσεις ασκήσεων

Προσπάθησε πρώτα να λύσεις τις ασκήσεις του **Τμήματος 21** χωρίς να κοιτάξεις παρακάτω.

### Άσκηση 1 — Τύποι γεωμετρίας

In [ ]:
municipal_unemployment.geometry.geom_type.value_counts()

### Άσκηση 2 — 10 υψηλότερα ποσοστά ανεργίας

In [ ]:
top_10_unemployment = municipal_unemployment.nlargest(
    10,
    "unemployment_rate",
)

top_10_unemployment[
    ["NAME_GR", "unemployment_rate", "geometry"]
]

### Άσκηση 3 — Χάρτης συνολικού πληθυσμού

In [ ]:
ax = municipal_unemployment.plot(
    column="population_total",
    cmap="Blues",
    legend=True,
    figsize=(10, 10),
    edgecolor="white",
    linewidth=0.25,
    legend_kwds={"label": "Population total"},
)

ax.set_title("Municipal population, Greece, 2021")
ax.set_axis_off()
plt.show()

### Άσκηση 4 — Join χωρίς indicator

In [ ]:
join_without_indicator = municipalities_join.merge(
    unemployment_join,
    on="municipality_code",
    how="left",
    validate="one_to_one",
)

join_without_indicator.head()

Χωρίς το `indicator=True` δεν έχουμε τη στήλη `_merge`, άρα χάνουμε τον άμεσο διαγνωστικό διαχωρισμό `left_only`, `right_only` και `both`. Μπορούμε ακόμη να ελέγξουμε missing values, αλλά ο έλεγχος είναι λιγότερο ρητός.

### Άσκηση 5 — Έκταση Δήμων

In [ ]:
municipal_areas = municipal_unemployment.copy()
municipal_areas["area_km2"] = municipal_areas.geometry.area / 1_000_000

municipal_areas.nlargest(5, "area_km2")[
    ["NAME_GR", "area_km2"]
]

Το EPSG:2100 είναι projected CRS με μονάδα το μέτρο. Επομένως το `.area` επιστρέφει τετραγωνικά μέτρα και η διαίρεση με `1_000_000` δίνει τετραγωνικά χιλιόμετρα. Η έκταση μιας προβολής εξακολουθεί να έχει παραμόρφωση, αλλά το Greek Grid είναι πολύ καταλληλότερο εδώ από geographic coordinates σε μοίρες.